# Waste Classification Tool

## Objective 5

Design and implement a locally runnable waste-classification tool that allows users to classify new waste images using the trained CNN model.

### Features

- Command-line prediction
- Desktop GUI (Tkinter)
- Confidence scores
- Score-CAM visual explanation

In [1]:
import os

import torch
import torch.nn.functional as F

from PIL import Image, ImageTk

import config
import tkinter as tk
from tkinter import filedialog, messagebox   
from dataset import get_clean_transforms
from models import build_model
from scorecam import ScoreCAM, overlay_heatmap

In [2]:
class WasteClassifier:

    def __init__(self, backbone_name):

        self.backbone_name = backbone_name
        self.device = config.DEVICE

        ckpt_path = os.path.join(
            config.CHECKPOINT_DIR,
            f"{backbone_name}_trashnet_best.pt"
        )

        if not os.path.exists(ckpt_path):
            raise FileNotFoundError(
                f"Checkpoint not found:\n{ckpt_path}"
            )

        self.model, self.target_layer = build_model(
            backbone_name,
            len(config.TRASHNET_CLASSES),
            pretrained=False
        )

        self.model.load_state_dict(
            torch.load(
                ckpt_path,
                map_location=self.device
            )
        )

        self.model = self.model.to(self.device)
        self.model.eval()

        _, self.eval_tf = get_clean_transforms()

        self.scorecam = ScoreCAM(
            self.model,
            self.target_layer,
            self.device
        )

In [3]:
@torch.no_grad()
def predict(self, pil_image):

    tensor = self.eval_tf(
        pil_image
    ).unsqueeze(0).to(self.device)

    logits = self.model(tensor)

    probs = F.softmax(logits, dim=1)[0]

    top_idx = int(probs.argmax())

    confidence = float(probs[top_idx])

    ranked = sorted(
        zip(
            config.TRASHNET_CLASSES,
            probs.tolist()
        ),
        key=lambda x: x[1],
        reverse=True
    )

    return {
        "predicted_class":
            config.TRASHNET_CLASSES[top_idx],

        "confidence":
            confidence,

        "all_probabilities":
            ranked
    }

WasteClassifier.predict = predict

In [4]:
def predict_with_explanation(
    self,
    pil_image,
    out_path
):

    tensor = self.eval_tf(
        pil_image
    ).unsqueeze(0)

    cam, class_idx = self.scorecam.generate(
        tensor
    )

    overlay = overlay_heatmap(
        pil_image.resize(
            (
                config.IMG_SIZE,
                config.IMG_SIZE
            )
        ),
        cam
    )

    import matplotlib.pyplot as plt

    plt.imsave(
        out_path,
        overlay
    )

    return (
        config.TRASHNET_CLASSES[class_idx],
        out_path
    )

WasteClassifier.predict_with_explanation = predict_with_explanation

In [5]:
def run_cli(backbone_name, image_path):

    classifier = WasteClassifier(backbone_name)

    pil_image = Image.open(
        image_path
    ).convert("RGB")

    result = classifier.predict(
        pil_image
    )

    print()

    print(
        "Predicted:",
        result["predicted_class"]
    )

    print(
        "Confidence:",
        f"{result['confidence']*100:.2f}%"
    )

    print()

    for cls, prob in result["all_probabilities"]:

        print(
            f"{cls:12s}",
            f"{prob*100:.2f}%"
        )

In [6]:
import tkinter as tk

from tkinter import (
    filedialog,
    messagebox
)

In [7]:
def run_gui(backbone_name):

    classifier = WasteClassifier(backbone_name)

    root = tk.Tk()

    root.title(
        "Waste Classification Tool"
    )

    root.geometry("480x620")

    image_label = tk.Label(
        root,
        text="No Image Loaded",
        bg="#eeeeee"
    )

    image_label.pack(
        pady=10,
        fill="x"
    )

    result_var = tk.StringVar(
        value="Choose an image."
    )

    tk.Label(
        root,
        textvariable=result_var,
        font=("Arial",14,"bold")
    ).pack()

    probs_text = tk.Text(
        root,
        height=8,
        width=45
    )

    probs_text.pack()

    current = {}

    def choose_photo():

        path = filedialog.askopenfilename(
            filetypes=[
                ("Images","*.jpg *.jpeg *.png")
            ]
        )

        if not path:
            return

        img = Image.open(path).convert("RGB")

        current["img"] = img

        preview = img.copy()

        preview.thumbnail((400,400))

        tk_img = ImageTk.PhotoImage(preview)

        current["tk"] = tk_img

        image_label.configure(image=tk_img)

        result = classifier.predict(img)

        result_var.set(
            f"{result['predicted_class']} "
            f"{result['confidence']*100:.1f}%"
        )

        probs_text.delete(
            "1.0",
            tk.END
        )

        for cls, prob in result["all_probabilities"]:

            probs_text.insert(
                tk.END,
                f"{cls:12s} {prob*100:.1f}%\n"
            )

    def show_scorecam():

        if "img" not in current:

            messagebox.showinfo(
                "Info",
                "Choose image first."
            )

            return

        out = os.path.join(
            config.RESULTS_DIR,
            "gui_scorecam.png"
        )

        _, path = classifier.predict_with_explanation(
            current["img"],
            out
        )

        explain = Image.open(path)

        explain.thumbnail((400,400))

        tk_img = ImageTk.PhotoImage(explain)

        current["cam"] = tk_img

        image_label.configure(image=tk_img)

    frame = tk.Frame(root)

    frame.pack(pady=10)

    tk.Button(
        frame,
        text="Choose Photo",
        command=choose_photo
    ).grid(row=0,column=0,padx=5)

    tk.Button(
        frame,
        text="Show ScoreCAM",
        command=show_scorecam
    ).grid(row=0,column=1,padx=5)

    root.mainloop()

In [8]:
BACKBONE = "resnet50"

In [9]:
IMAGE_PATH = os.path.join("data", "trashnet", "cardboard", "cardboard1.jpg")

run_cli(BACKBONE, IMAGE_PATH)


Predicted: cardboard
Confidence: 99.99%

cardboard    99.99%
glass        0.00%
paper        0.00%
metal        0.00%
plastic      0.00%
trash        0.00%


In [10]:
import os

print(os.listdir("checkpoints"))

['efficientnet_b0_trashnet_best.pt', 'mobilenet_v3_large_trashnet_best.pt', 'resnet50_dg_both_best.pt', 'resnet50_trashnet_best.pt']


In [12]:
run_gui(BACKBONE)